In [1]:
import pandas as pd
import json

In [2]:
input_path = "../data/arxiv-metadata-oai-snapshot.json"
output_path = "../data/papers_clean.jsonl"

read in chunks → keep wanted columns → keep wanted categories → clean fields → drop bad rows → deduplicate → save clean file

In [3]:
keep_cols = ["id", "title", "abstract", "authors", "categories", "update_date"]
pattern = r"cs\.LG|cs\.CL|cs\.IR|q-bio\.NC|q-bio\.QM|q-bio\.GN|q-bio\.MN"

chunksize = 10000

with open(output_path, "w", encoding="utf-8") as out:
    for chunk in pd.read_json(input_path, lines=True, chunksize=chunksize):
        chunk = chunk[keep_cols].copy()

        chunk["title"] = chunk["title"].fillna("").str.strip()
        chunk["abstract"] = chunk["abstract"].fillna("").str.strip()
        chunk["authors"] = chunk["authors"].fillna("").str.strip()
        chunk["categories"] = chunk["categories"].fillna("").str.strip()
        chunk["update_date"] = chunk["update_date"].fillna("").str.strip()

        chunk = chunk[chunk["categories"].str.contains(pattern, na=False)]
        chunk = chunk[(chunk["title"] != "") & (chunk["abstract"] != "")]
        chunk = chunk[chunk["abstract"].str.len() >= 50]

        chunk = chunk.drop_duplicates(subset=["id"])

        for record in chunk.to_dict(orient="records"):
            out.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Done")

Done


Second deduplication pass on the full cleaned file, because duplicates can appear across different chunks, and drop_duplicates inside a chunk only removes duplicates within that chunk

- Dedup by ID
- Dedup by title & abstract

In [4]:
input_path = "../data/papers_clean.jsonl"
output_path = "../data/papers_final.jsonl"

In [5]:
df = pd.read_json(input_path, lines=True)

print("Initial rows:", len(df))

df["id"] = df["id"].fillna("").astype(str).str.strip()
df["title"] = df["title"].fillna("").str.strip()
df["abstract"] = df["abstract"].fillna("").str.strip()

df = df.drop_duplicates(subset=["id"])
print("After ID dedup:", len(df))

df = df.drop_duplicates(subset=["title", "abstract"])
print("After title+abstract dedup:", len(df))

with open(output_path, "w", encoding="utf-8") as f:
    for record in df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved final file to {output_path}")

Initial rows: 375714
After ID dedup: 375714
After title+abstract dedup: 375672
Saved final file to ../data/papers_final.jsonl


Check

In [6]:
df_final = pd.read_json("../data/papers_final.jsonl", lines=True)

print(df_final.shape)
print(df_final[["id", "title", "categories"]].head())

(375672, 6)
         id                                              title  \
0  704.0021  Molecular Synchronization Waves in Arrays of A...   
1  704.0036  A remark on the number of steady states in a m...   
2  704.0392  Simulation of Robustness against Lesions of Co...   
3  704.0634  A Finite Element framework for computation of ...   
4  704.0648  Behavioral response to strong aversive stimuli...   

                                categories  
0         nlin.PS physics.chem-ph q-bio.MN  
1                        q-bio.QM q-bio.MN  
2  q-bio.NC cond-mat.dis-nn physics.soc-ph  
3                        q-bio.BM q-bio.QM  
4                                 q-bio.NC  
